# WolBanking77 : Classification d'intentions bancaires en Wolof

Ce notebook unique realise le projet complet : exploration des donnees, baseline Machine Learning, entrainement XLM-RoBERTa, evaluation et comparaison.

## Architecture d'execution

- **GitHub** fournit le projet, le code et les CSV. Colab clone le depot a chaque execution dans `/content/WolBanking77`.
- **Google Drive** conserve les sorties persistantes : meilleur checkpoint, rapports CSV/JSON et graphiques.
- **Le test officiel reste reserve a l'evaluation finale**. Le meilleur checkpoint Transformer est choisi avec 10 % du train, separes de maniere stratifiee.

Avant de lancer le notebook, placez les donnees dans votre depot GitHub et renseignez son URL HTTPS dans la cellule suivante. Pour une explication complete, consultez `docs/GUIDE_COLAB.md` dans le depot.

## 1. Importer le projet depuis GitHub et connecter Google Drive

In [ ]:
from google.colab import drive
from pathlib import Path

# REMPLACEZ par l'URL HTTPS de votre depot GitHub.
GITHUB_REPOSITORY_URL = 'https://github.com/VOTRE_COMPTE/WolBanking77.git'
REPOSITORY_DIR = Path('/content/WolBanking77')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/WolBanking77_runs')

drive.mount('/content/drive')
if not REPOSITORY_DIR.exists():
    !git clone $GITHUB_REPOSITORY_URL $REPOSITORY_DIR
else:
    %cd $REPOSITORY_DIR
    !git pull
%cd $REPOSITORY_DIR

assert (REPOSITORY_DIR / 'data').exists(), 'Le depot GitHub doit contenir le dossier data.'
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Projet GitHub :', REPOSITORY_DIR)
print('Sorties Drive :', DRIVE_OUTPUT_DIR)

## 2. Installer les bibliotheques et configurer l'experience

Choisissez `5k_split` pour le jeu de 4 000 / 1 000 exemples ou `full` pour le jeu complet. Activez un GPU Colab pour accelerer le Transformer.

In [ ]:
!pip -q install transformers sentencepiece joblib

import json
import random
import re

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

DATASET_NAME = '5k_split'
TEXT_COLUMN, LABEL_COLUMN = 'input_wo', 'label'
MODEL_NAME, MAX_LENGTH = 'xlm-roberta-base', 128
BATCH_SIZE = 16 if torch.cuda.is_available() else 4
LEARNING_RATE, EPOCHS, PATIENCE = 2e-5, 6, 2
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

DATA_DIR = REPOSITORY_DIR / 'data' / DATASET_NAME
MODELS_DIR, REPORTS_DIR = DRIVE_OUTPUT_DIR / 'models', DRIVE_OUTPUT_DIR / 'reports'
MODELS_DIR.mkdir(exist_ok=True); REPORTS_DIR.mkdir(exist_ok=True)
print('Device :', device)
print('Donnees :', DATA_DIR)

## 3. Fonction `clean_text`

Normalise les espaces sans supprimer les lettres et diacritiques utiles du wolof.

In [ ]:
def clean_text(text):
    return re.sub(r'\s+', ' ', str(text).strip())

## 4. Fonction `load_data`

Charge les deux partitions CSV, controle les textes et transforme les 77 intentions en identifiants numeriques pour le Transformer.

In [ ]:
def load_data():
    train = pd.read_csv(DATA_DIR / 'train' / 'train.csv', encoding='utf-8')
    test = pd.read_csv(DATA_DIR / 'test' / 'test.csv', encoding='utf-8')
    for frame in (train, test):
        frame.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN], inplace=True)
        frame['text'] = frame[TEXT_COLUMN].map(clean_text)
        frame.drop(frame.index[frame['text'].eq('')], inplace=True)
        frame.drop_duplicates(subset=['text'], inplace=True)
        frame.reset_index(drop=True, inplace=True)
    encoder = LabelEncoder()
    train['label_id'] = encoder.fit_transform(train[LABEL_COLUMN])
    test['label_id'] = encoder.transform(test[LABEL_COLUMN])
    return train, test, encoder

train_df, test_df, label_encoder = load_data()
print(f'Train : {len(train_df)} | Test : {len(test_df)} | Intentions : {len(label_encoder.classes_)}')
display(train_df[['input_wo', 'label']].head())

## 5. Fonction `explore_data`

Produit un controle de qualite et la distribution des intentions, utiles pour le rapport de projet.

In [ ]:
def explore_data(train, test):
    summary = pd.DataFrame({'train': [len(train), train['label'].nunique(), train['text'].duplicated().sum()], 'test': [len(test), test['label'].nunique(), test['text'].duplicated().sum()]}, index=['Exemples', 'Intentions', 'Doublons'])
    display(summary)
    counts = train['label'].value_counts().sort_values()
    fig, axis = plt.subplots(figsize=(10, 16))
    counts.plot.barh(ax=axis, color='#197278')
    axis.set(title='Repartition des intentions', xlabel='Nombre d exemples', ylabel='Intention')
    plt.tight_layout(); plt.savefig(REPORTS_DIR / f'{DATASET_NAME}_distribution.png', dpi=160); plt.show()

explore_data(train_df, test_df)

## 6. Fonction `build_baseline`

Construit la reference Machine Learning. Les n-grammes de caracteres aident a capturer les variations orthographiques du wolof.

In [ ]:
def build_baseline():
    features = FeatureUnion([
        ('words', TfidfVectorizer(analyzer='word', ngram_range=(1, 2), sublinear_tf=True, max_features=60_000)),
        ('chars', TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), sublinear_tf=True, max_features=80_000)),
    ])
    return Pipeline([('features', features), ('classifier', LinearSVC(C=1.0))])

baseline = build_baseline()
baseline.fit(train_df['text'], train_df['label'])
baseline_predictions = baseline.predict(test_df['text'])
joblib.dump(baseline, MODELS_DIR / f'baseline_{DATASET_NAME}.joblib')

## 7. Fonction `evaluate_model`

Calcule les metriques, enregistre un rapport CSV/JSON et dessine la matrice de confusion dans Drive.

In [ ]:
def evaluate_model(y_true, y_pred, model_name, prefix):
    metrics = {'model': model_name, 'dataset': DATASET_NAME, 'accuracy': float(accuracy_score(y_true, y_pred)), 'macro_f1': float(f1_score(y_true, y_pred, average='macro', zero_division=0))}
    print(metrics)
    with open(REPORTS_DIR / f'{prefix}_metrics_{DATASET_NAME}.json', 'w', encoding='utf-8') as file: json.dump(metrics, file, ensure_ascii=False, indent=2)
    report = pd.DataFrame(classification_report(y_true, y_pred, output_dict=True, zero_division=0)).T
    report.to_csv(REPORTS_DIR / f'{prefix}_report_{DATASET_NAME}.csv', encoding='utf-8')
    matrix = confusion_matrix(y_true, y_pred, labels=label_encoder.classes_)
    fig, axis = plt.subplots(figsize=(18, 15))
    sns.heatmap(matrix, cmap='Blues', xticklabels=label_encoder.classes_, yticklabels=label_encoder.classes_, ax=axis)
    axis.set(title=f'Matrice de confusion - {model_name}', xlabel='Prediction', ylabel='Reel')
    plt.tight_layout(); plt.savefig(REPORTS_DIR / f'{prefix}_confusion_{DATASET_NAME}.png', dpi=180); plt.show()
    return metrics, report

baseline_metrics, baseline_report = evaluate_model(test_df['label'], baseline_predictions, 'TF-IDF + LinearSVC', 'baseline')

## 8. Classe `IntentDataset` et fonction `create_dataloaders`

Tokenise les textes pour XLM-RoBERTa et cree un sous-ensemble de validation stratifie. Le test officiel n'est jamais utilise pour choisir le checkpoint.

In [ ]:
class IntentDataset(Dataset):
    def __init__(self, frame, tokenizer):
        self.texts, self.labels = frame['text'].tolist(), frame['label_id'].tolist()
        self.tokenizer = tokenizer
    def __len__(self): return len(self.labels)
    def __getitem__(self, index):
        tokens = self.tokenizer(self.texts[index], truncation=True, max_length=MAX_LENGTH, padding='max_length', return_tensors='pt')
        return {'input_ids': tokens['input_ids'].flatten(), 'attention_mask': tokens['attention_mask'].flatten(), 'labels': torch.tensor(self.labels[index], dtype=torch.long)}

def create_dataloaders():
    train_part, val_part = train_test_split(train_df, test_size=0.10, stratify=train_df['label_id'], random_state=SEED)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    make_loader = lambda frame, shuffle=False: DataLoader(IntentDataset(frame, tokenizer), batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=2, pin_memory=torch.cuda.is_available())
    return tokenizer, make_loader(train_part, True), make_loader(val_part), make_loader(test_df)

tokenizer, train_loader, val_loader, test_loader = create_dataloaders()

## 9. Fonction `run_epoch`

Execute une epoque d'entrainement ou d'evaluation et retourne les predictions, loss, accuracy et macro-F1.

In [ ]:
def run_epoch(model, loader, optimizer=None, scheduler=None):
    training = optimizer is not None
    model.train() if training else model.eval()
    loss_sum, predictions, labels = 0.0, [], []
    for batch in loader:
        inputs = {key: value.to(device) for key, value in batch.items()}
        if training: optimizer.zero_grad()
        with torch.set_grad_enabled(training):
            output = model(**inputs)
            if training:
                output.loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step(); scheduler.step()
        loss_sum += output.loss.item()
        predictions.extend(output.logits.argmax(dim=1).detach().cpu().tolist())
        labels.extend(inputs['labels'].detach().cpu().tolist())
    return {'loss': loss_sum / len(loader), 'accuracy': accuracy_score(labels, predictions), 'macro_f1': f1_score(labels, predictions, average='macro', zero_division=0), 'predictions': predictions, 'labels': labels}

## 10. Fonction `train_transformer`

Fine-tune XLM-RoBERTa. Lorsqu'un macro-F1 de validation s'ameliore, le checkpoint est immediatement enregistre dans Google Drive. Il restera donc disponible meme si la session Colab est interrompue.

In [ ]:
def train_transformer():
    id2label = {index: label for index, label in enumerate(label_encoder.classes_)}
    label2id = {label: index for index, label in id2label.items()}
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=len(label_encoder.classes_), id2label=id2label, label2id=label2id).to(device)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    total_steps = len(train_loader) * EPOCHS
    scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(0.1 * total_steps), num_training_steps=total_steps)
    model_dir = MODELS_DIR / f'xlmr_{DATASET_NAME}'
    model_dir.mkdir(exist_ok=True)
    history = {'train_loss': [], 'val_loss': [], 'train_f1': [], 'val_f1': []}
    best_f1, stale = -1.0, 0
    for epoch in range(EPOCHS):
        train_metrics = run_epoch(model, train_loader, optimizer, scheduler)
        val_metrics = run_epoch(model, val_loader)
        for key, value in [('train_loss', train_metrics['loss']), ('val_loss', val_metrics['loss']), ('train_f1', train_metrics['macro_f1']), ('val_f1', val_metrics['macro_f1'])]: history[key].append(value)
        print(f"Epoch {epoch + 1}/{EPOCHS} | train F1={train_metrics['macro_f1']:.4f} | validation F1={val_metrics['macro_f1']:.4f}")
        if val_metrics['macro_f1'] > best_f1:
            best_f1, stale = val_metrics['macro_f1'], 0
            model.save_pretrained(model_dir)
            tokenizer.save_pretrained(model_dir)
            with open(model_dir / 'labels.json', 'w', encoding='utf-8') as file: json.dump(label_encoder.classes_.tolist(), file, ensure_ascii=False, indent=2)
            print('Checkpoint sauvegarde dans Drive :', model_dir)
        else:
            stale += 1
            if stale >= PATIENCE: print('Arret anticipe.'); break
    return model_dir, history

transformer_dir, history = train_transformer()

## 11. Courbes d'apprentissage et evaluation finale du Transformer

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history['train_loss'], label='Train'); axes[0].plot(history['val_loss'], label='Validation'); axes[0].set(title='Loss', xlabel='Epoch'); axes[0].legend()
axes[1].plot(history['train_f1'], label='Train'); axes[1].plot(history['val_f1'], label='Validation'); axes[1].set(title='Macro-F1', xlabel='Epoch', ylim=(0, 1)); axes[1].legend()
plt.tight_layout(); plt.savefig(REPORTS_DIR / f'xlmr_history_{DATASET_NAME}.png', dpi=160); plt.show()

transformer_model = AutoModelForSequenceClassification.from_pretrained(transformer_dir).to(device)
transformer_test = run_epoch(transformer_model, test_loader)
transformer_predictions = label_encoder.inverse_transform(transformer_test['predictions'])
transformer_metrics, transformer_report = evaluate_model(test_df['label'], transformer_predictions, 'XLM-RoBERTa', 'transformer')

## 12. Comparaison finale

Le macro-F1 est la mesure principale : il donne le meme poids a chacune des 77 intentions.

In [ ]:
comparison = pd.DataFrame([baseline_metrics, transformer_metrics]).set_index('model')[['accuracy', 'macro_f1']]
display(comparison.style.format('{:.2%}'))
plot_data = comparison.reset_index().melt(id_vars='model', var_name='Mesure', value_name='Score')
sns.barplot(data=plot_data, x='Mesure', y='Score', hue='model')
plt.ylim(0, 1); plt.title('Comparaison des modeles'); plt.tight_layout(); plt.savefig(REPORTS_DIR / f'comparison_{DATASET_NAME}.png', dpi=160); plt.show()

## 13. Fonction `predict_intent`

Utilisez cette fonction pour classer une nouvelle question bancaire en wolof avec le meilleur checkpoint enregistre dans Drive.

In [ ]:
def predict_intent(text):
    transformer_model.eval()
    tokens = tokenizer(clean_text(text), truncation=True, max_length=MAX_LENGTH, padding='max_length', return_tensors='pt')
    with torch.no_grad():
        logits = transformer_model(input_ids=tokens['input_ids'].to(device), attention_mask=tokens['attention_mask'].to(device)).logits
    probabilities = torch.softmax(logits, dim=1)[0]
    index = int(probabilities.argmax())
    return {'intention': label_encoder.inverse_transform([index])[0], 'confiance': float(probabilities[index])}

predict_intent('Dama bëgg xam xaalis bi nekk ci sama kont.')